In [1]:
import numpy as np
import pandas as pd
from statsmodels.datasets import grunfeld
'''
data = grunfeld.load_pandas().data
data.year = data.year.astype(np.int64)

# Establish unique IDs to conform with package
N = len(np.unique(data.firm))
ID = dict(zip(np.unique(data.firm).tolist(),np.arange(1,N+1)))
data.firm = data.firm.apply(lambda x: ID[x])

# use multi-index for panel groups
data = data.set_index(['firm', 'year'])
y = data['invest']
X = data.drop('invest', axis=1)

# Call ipca
from ipca import InstrumentedPCA
regr = InstrumentedPCA(n_factors=1, intercept=False)
regr = regr.fit(X=X, y=y)
Gamma, Factors = regr.get_factors(label_ind=True)
'''

"\ndata = grunfeld.load_pandas().data\ndata.year = data.year.astype(np.int64)\n\n# Establish unique IDs to conform with package\nN = len(np.unique(data.firm))\nID = dict(zip(np.unique(data.firm).tolist(),np.arange(1,N+1)))\ndata.firm = data.firm.apply(lambda x: ID[x])\n\n# use multi-index for panel groups\ndata = data.set_index(['firm', 'year'])\ny = data['invest']\nX = data.drop('invest', axis=1)\n\n# Call ipca\nfrom ipca import InstrumentedPCA\nregr = InstrumentedPCA(n_factors=1, intercept=False)\nregr = regr.fit(X=X, y=y)\nGamma, Factors = regr.get_factors(label_ind=True)\n"

In [2]:
from ipca import InstrumentedPCA
import numpy as np
import pandas as pd
data = pd.read_feather('chars_raw_imputed.feather')
data['size'] = abs(data['prc'] * data['shrout'])
data = data.set_index(['permno', 'date'])

In [3]:
y = data['ret']
X = data[['abr','absacc','acc','adm','age','agr','alm','ato','baspread','beta','size']]

# Call ipca
from ipca import InstrumentedPCA
regr = InstrumentedPCA(n_factors=1, intercept=False)
regr = regr.fit(X=X, y=y)
Gamma, Factors = regr.get_factors(label_ind=True)

full_fitted = pd.Series(index=X.index, dtype=float, name='fitted')

X_pred = X.loc[~X.isna().any(axis=1)]

pre_y = regr.predict(
    X=X_pred,
    mean_factor=False,
    data_type='panel',
    label_ind=False
)

full_fitted.loc[X_pred.index] = pre_y.ravel()
full_fitted

The panel dimensions are:
n_samples: 22222 , L: 11 , T: 638


[========================================================================] 100%


Step 1 - Aggregate Update: 1148487.2635695173
Step 2 - Aggregate Update: 0.5974776652615104
Step 3 - Aggregate Update: 0.40605230622209726
Step 4 - Aggregate Update: 0.09484755936401046
Step 5 - Aggregate Update: 0.020491260075643725
Step 6 - Aggregate Update: 0.004553442378494976
Step 7 - Aggregate Update: 0.0010398877605234391
Step 8 - Aggregate Update: 0.0002414498269203147
Step 9 - Aggregate Update: 5.667989979041277e-05
Step 10 - Aggregate Update: 1.3411118553685597e-05
Step 11 - Aggregate Update: 3.1923488054141913e-06
-- Convergence Reached --


permno  date      
10000   1986-03-31    0.023889
        1986-04-30    0.004943
        1986-05-31    0.008566
        1986-06-30    0.001563
        1986-07-31   -0.015943
                        ...   
93436   2024-08-31   -0.032917
        2024-09-30   -0.000131
        2024-10-31    0.000414
        2024-11-30    0.042141
        2024-12-31   -0.007159
Name: fitted, Length: 3028230, dtype: float64

In [9]:
Factors

,1971-11-30,1971-12-31,1972-01-31,1972-02-29,1972-03-31,1972-04-30,1972-05-31,1972-06-30,1972-07-31,1972-08-31,...,2016-03-31,2016-04-30,2016-05-31,2016-06-30,2016-07-31,2016-08-31,2016-09-30,2016-10-31,2016-11-30,2016-12-31
0,-0.099881,0.322868,0.336423,0.122755,-0.009207,0.006406,-0.056475,-0.095081,-0.088434,0.032041,...,0.241982,0.102061,-0.014512,-0.005215,0.144536,0.065632,0.069318,-0.153718,0.226254,0.028928


In [10]:
import datetime as dt
from ipca import InstrumentedPCA

data_is = data[data.index.get_level_values('date') <= pd.Timestamp('2016-12-31')]
data_oos = data[data.index.get_level_values('date') > pd.Timestamp('2016-12-31')]
y_is = data_is['ret']
X_is = data_is[['abr','absacc','acc','adm','age','agr','alm','ato','baspread','beta','size']]
y_oos = data_oos['ret']
X_oos = data_oos[['abr','absacc','acc','adm','age','agr','alm','ato','baspread','beta','size']]

# Call ipca
regr = InstrumentedPCA(n_factors=1, intercept=False)
regr = regr.fit(X=X_is, y=y_is)
Gamma, Factors = regr.get_factors(label_ind=True)

full_fitted = pd.Series(index=X_oos.index, dtype=float, name='fitted')

X_pred = X_oos.loc[~X_oos.isna().any(axis=1)]

pre_y = regr.predict(
    X=X_pred,
    mean_factor=True,
    data_type='panel',
    label_ind=False
)

full_fitted.loc[X_pred.index] = pre_y.ravel()
full_fitted = pd.merge(full_fitted, y_oos, right_index=True, left_index=True, how ='outer')

The panel dimensions are:
n_samples: 20293 , L: 11 , T: 542


[========================================================================] 100%


Step 1 - Aggregate Update: 521664.7874712139
Step 2 - Aggregate Update: 0.6044681344898455
Step 3 - Aggregate Update: 0.29149180963111665
Step 4 - Aggregate Update: 0.06253690269490086
Step 5 - Aggregate Update: 0.012493646377294776
Step 6 - Aggregate Update: 0.0026093838421219484
Step 7 - Aggregate Update: 0.0005650397624616943
Step 8 - Aggregate Update: 0.00012583681955302684
Step 9 - Aggregate Update: 2.867072137602289e-05
Step 10 - Aggregate Update: 6.657464930004409e-06
-- Convergence Reached --


In [ ]:
import pandas as pd
import numpy as np
from ipca import InstrumentedPCA

print("Loading data...")
data = pd.read_feather('chars_raw_imputed.feather')

data['size'] = abs(data['prc'] * data['shrout'])
data = data.set_index(['permno', 'date'])

# Split
data_is = data[data.index.get_level_values('date') <= pd.Timestamp('2016-12-31')]
data_oos = data[data.index.get_level_values('date') > pd.Timestamp('2016-12-31')]

y_is = data_is['ret']
X_is = data_is[['abr','absacc','acc','adm','age','agr','alm','ato','baspread','beta','size']]

y_oos = data_oos['ret']
X_oos = data_oos[['abr','absacc','acc','adm','age','agr','alm','ato','baspread','beta','size']]

print("Fitting IPCA...")
regr = InstrumentedPCA(n_factors=1, intercept=False)
regr = regr.fit(X=X_is, y=y_is)

print("Predicting...")
X_pred = X_oos.loc[~X_oos.isna().any(axis=1)]

pre_y = regr.predict(
    X=X_pred,
    mean_factor=True,
    data_type='panel',
    label_ind=False
)

full_fitted = pd.Series(index=X_oos.index, dtype=float, name='fitted')
full_fitted.loc[X_pred.index] = pre_y.ravel()

result = pd.merge(full_fitted, y_oos, right_index=True, left_index=True, how='outer')

print("Saving output...")
result.to_feather("ipca_results.feather")

print("Done.")